In [6]:
import subprocess, time, requests, os

def ollama_repond():
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not os.path.exists("/usr/local/bin/ollama"):
    print("Installation des prérequis et d'Ollama (~2 min)...")
    subprocess.run("apt-get install -y -qq zstd pciutils lshw", shell=True)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

if not ollama_repond():
    print("Démarrage du serveur...")
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                     start_new_session=True)
    for _ in range(30):
        if ollama_repond():
            break
        time.sleep(1)

print("Ollama :", "actif" if ollama_repond() else "ARRÊTÉ")

Ollama : actif


In [7]:
!ollama pull qwen2.5:7b
!ollama list


NAME          ID              SIZE      MODIFIED               
qwen2.5:7b    845dbda0ea48    4.7 GB    Less than a second ago    


In [9]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
DOSSIER = "/content/drive/MyDrive/stage-graphrag"
os.makedirs(DOSSIER, exist_ok=True)

NECESSAIRES = ["corpus.json", "jeu_de_test.json",
               "classement_vectoriel.json", "classement_graphe.json"]

for nom in NECESSAIRES:
    if os.path.exists(f"/content/{nom}"):
        shutil.copy(f"/content/{nom}", f"{DOSSIER}/{nom}")
        print("copié :", nom)

presents = os.listdir(DOSSIER)
for nom in NECESSAIRES:
    print(f"{nom:<28}", "présent" if nom in presents else "ABSENT")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
copié : corpus.json
copié : jeu_de_test.json
copié : classement_vectoriel.json
copié : classement_graphe.json
corpus.json                  présent
jeu_de_test.json             présent
classement_vectoriel.json    présent
classement_graphe.json       présent


In [10]:
import json, time, requests

K = 5
DOSSIER = "/content/drive/MyDrive/stage-graphrag"

PROMPT = """You are a biomedical question-answering system.

Answer the question using ONLY the abstracts provided below.

RULES:
1. Base your answer solely on the abstracts. Never use outside knowledge.
2. Answer with the specific entity name requested, followed by one short
   sentence of justification.
3. If the abstracts do not contain the answer, reply exactly:
   NOT FOUND IN THE PROVIDED ABSTRACTS

ABSTRACTS:
{contexte}

QUESTION: {question}

ANSWER:"""

def charger(nom):
    with open(f"{DOSSIER}/{nom}", encoding="utf-8") as fichier:
        return json.load(fichier)

corpus = {a["pmid"]: a for a in charger("corpus.json")}
questions = charger("jeu_de_test.json")
classements = {
    "vectoriel": {l["id"]: l["classement"] for l in charger("classement_vectoriel.json")},
    "graphe":    {l["id"]: l["classement"] for l in charger("classement_graphe.json")},
}

def generer(question, pmids, essais=3):
    blocs = []
    for i, pmid in enumerate(pmids, 1):
        a = corpus.get(pmid)
        if a:
            blocs.append(f"[{i}] PMID {pmid} — {a['titre']}\n{a['resume']}")
    contexte = "\n\n".join(blocs) if blocs else "(no abstracts retrieved)"
    for tentative in range(essais):
        try:
            r = requests.post(
                "http://127.0.0.1:11434/api/generate",
                json={"model": "qwen2.5:7b",
                      "prompt": PROMPT.format(contexte=contexte,
                                              question=question),
                      "stream": False,
                      "options": {"temperature": 0, "num_ctx": 8192,
                                  "num_predict": 200}},
                timeout=600)
            return r.json()["response"].strip(), contexte
        except Exception as e:
            if tentative == essais - 1:
                raise
            time.sleep(5)

generer("Warm up.", [])          # chauffe, hors chronomètre
print("Modèle chargé.\n")

reponses = []
for q in questions:
    for pipeline in ("vectoriel", "graphe"):
        pmids = classements[pipeline].get(q["id"], [])[:K]
        t0 = time.time()
        texte, contexte = generer(q["question"], pmids)
        reponses.append({
            "id": q["id"], "type": q["type"], "pipeline": pipeline,
            "question": q["question"],
            "reponses_acceptables": q["reponses_acceptables"],
            "pmids_fournis": pmids, "pmids_attendus": q.get("pmids", []),
            "reponse": texte, "latence": round(time.time() - t0, 2),
            "contexte": contexte,
        })
    print(f"{q['id']} ✓")

with open(f"{DOSSIER}/reponses_generees.json", "w",
          encoding="utf-8") as fichier:
    json.dump(reponses, fichier, ensure_ascii=False, indent=2)

print(f"\n{len(reponses)} réponses générées.")
for pipeline in ("vectoriel", "graphe"):
    lat = [r["latence"] for r in reponses if r["pipeline"] == pipeline]
    print(f"  {pipeline:<12} latence moyenne : {sum(lat) / len(lat):.1f} s")

Modèle chargé.

MH01 ✓
MH02 ✓
MH03 ✓
MH04 ✓
MH05 ✓
MH06 ✓
MH07 ✓
MH08 ✓
MH09 ✓
MH10 ✓
MH11 ✓
SH01 ✓
SH02 ✓
SH03 ✓
SH04 ✓
SH05 ✓
SH06 ✓
SH07 ✓
SH08 ✓
SH09 ✓

40 réponses générées.
  vectoriel    latence moyenne : 2.8 s
  graphe       latence moyenne : 2.9 s


In [12]:
import json, re
from collections import defaultdict

with open(f"{DOSSIER}/reponses_generees.json", encoding="utf-8") as fichier:
    reponses = json.load(fichier)

def simplifie(t):
    return re.sub(r"[^a-z0-9]+", " ", t.lower()).strip()

for r in reponses:
    texte = simplifie(r["reponse"])
    r["correct"] = any(simplifie(a) in texte
                       for a in r["reponses_acceptables"])
    r["abstention"] = "NOT FOUND" in r["reponse"].upper()

print(f"{'':<14}{'exactitude':>12}{'abstention':>13}{'latence':>10}")
for pipeline in ("vectoriel", "graphe"):
    sous = [r for r in reponses if r["pipeline"] == pipeline]
    exact = sum(r["correct"] for r in sous) / len(sous)
    abst = sum(r["abstention"] for r in sous) / len(sous)
    lat = sum(r["latence"] for r in sous) / len(sous)
    print(f"{pipeline:<14}{exact:>12.2f}{abst:>13.2f}{lat:>9.1f}s")

print("\n=== PAR CATÉGORIE ===")
for genre in ("single-hop", "pont", "intersection"):
    print(f"\n{genre}")
    for pipeline in ("vectoriel", "graphe"):
        sous = [r for r in reponses
                if r["pipeline"] == pipeline and r["type"] == genre]
        if sous:
            e = sum(r["correct"] for r in sous) / len(sous)
            print(f"  {pipeline:<12} {e:.2f}  ({len(sous)} questions)")

print("\n=== DÉTAIL ===")
par_id = defaultdict(dict)
for r in reponses:
    par_id[r["id"]][r["pipeline"]] = r
for ident, paire in par_id.items():
    v, g = paire.get("vectoriel"), paire.get("graphe")
    marque = lambda r: "OK" if r["correct"] else ("--" if r["abstention"] else "XX")
    print(f"\n{ident} [{v['type']}]  vectoriel {marque(v)}   graphe {marque(g)}")
    print(f"  attendu : {', '.join(v['reponses_acceptables'])}")
    print(f"  vect    : {v['reponse'][:88]}")
    print(f"  graphe  : {g['reponse'][:88]}")

with open(f"{DOSSIER}/evaluation_exactitude.json", "w",
          encoding="utf-8") as fichier:
    json.dump(reponses, fichier, ensure_ascii=False, indent=2)
print("\nRésultats enregistrés.")

                exactitude   abstention   latence
vectoriel             0.30         0.50      2.8s
graphe                0.70         0.15      2.9s

=== PAR CATÉGORIE ===

single-hop
  vectoriel    0.44  (9 questions)
  graphe       0.78  (9 questions)

pont
  vectoriel    0.00  (3 questions)
  graphe       0.67  (3 questions)

intersection
  vectoriel    0.25  (8 questions)
  graphe       0.62  (8 questions)

=== DÉTAIL ===

MH01 [pont]  vectoriel --   graphe --
  attendu : nor-imatinib/imatinib metabolic ratio, nor-imatinib/imatinib ratio
  vect    : NOT FOUND IN THE PROVIDED ABSTRACTS
  graphe  : NOT FOUND IN THE PROVIDED ABSTRACTS

MH02 [pont]  vectoriel XX   graphe OK
  attendu : PI3 kinase pathway, PI3K pathway, PI3 kinase (PI3K) pathway
  vect    : Gi-coupled receptor pathway

Justification: Abstract [3] states that the P2Y 12 receptor
  graphe  : Gi-coupled receptor pathway. The abstract states that P2Y 12 receptor, which cangrelor a

MH03 [pont]  vectoriel --   graphe OK
  a

In [8]:
!rm -f /content/classement_graphe.json

In [13]:
JUGE_FIDELITE = """You are evaluating whether an ANSWER is supported by the
provided ABSTRACTS.

Score from 0 to 4:
4 = every claim in the answer is directly stated in the abstracts
3 = the main claim is stated; minor details are not
2 = the main claim is partially supported
1 = the answer goes well beyond what the abstracts state
0 = the answer contradicts the abstracts, or nothing in them supports it

If the answer states that the information was not found, and it is indeed
absent from the abstracts, score 4.

ABSTRACTS:
{contexte}

ANSWER:
{reponse}

Reply with a single digit from 0 to 4, nothing else."""

JUGE_PERTINENCE = """You are evaluating whether an ANSWER addresses the QUESTION.

Score from 0 to 4:
4 = the answer directly and completely addresses the question
3 = the answer addresses the question but is incomplete
2 = the answer is partially on topic
1 = the answer is mostly off topic
0 = the answer does not address the question at all

An answer stating that the information was not found addresses the question
honestly but does not answer it: score 1.

QUESTION:
{question}

ANSWER:
{reponse}

Reply with a single digit from 0 to 4, nothing else."""

print("Prompts de jugement définis.")

Prompts de jugement définis.


In [14]:
import json, re, time, requests
from collections import defaultdict

DOSSIER = "/content/drive/MyDrive/stage-graphrag"
REPETITIONS = 3

with open(f"{DOSSIER}/evaluation_exactitude.json", encoding="utf-8") as fichier:
    reponses = json.load(fichier)

def juger(prompt, essais=3):
    for tentative in range(essais):
        try:
            r = requests.post(
                "http://127.0.0.1:11434/api/generate",
                json={"model": "qwen2.5:7b", "prompt": prompt, "stream": False,
                      "options": {"temperature": 0, "num_ctx": 8192,
                                  "num_predict": 5}},
                timeout=600)
            texte = r.json()["response"].strip()
            chiffres = re.findall(r"[0-4]", texte)
            return int(chiffres[0]) / 4.0 if chiffres else None
        except Exception:
            if tentative == essais - 1:
                return None
            time.sleep(5)

juger("Reply with the digit 4.")          # chauffe
print("Modèle chargé.\n")

for i, r in enumerate(reponses, 1):
    fid, per = [], []
    for _ in range(REPETITIONS):
        fid.append(juger(JUGE_FIDELITE.format(contexte=r["contexte"],
                                              reponse=r["reponse"])))
        per.append(juger(JUGE_PERTINENCE.format(question=r["question"],
                                                reponse=r["reponse"])))
    fid = [x for x in fid if x is not None]
    per = [x for x in per if x is not None]
    r["fidelite"] = sum(fid) / len(fid) if fid else None
    r["pertinence"] = sum(per) / len(per) if per else None
    r["fidelite_ecart"] = max(fid) - min(fid) if len(fid) > 1 else 0
    r["pertinence_ecart"] = max(per) - min(per) if len(per) > 1 else 0
    if i % 10 == 0:
        print(f"  {i}/{len(reponses)}")

with open(f"{DOSSIER}/evaluation_ragas.json", "w",
          encoding="utf-8") as fichier:
    json.dump(reponses, fichier, ensure_ascii=False, indent=2)

print(f"\n{'':<14}{'fidélité':>11}{'pertinence':>13}{'exactitude':>13}")
for pipeline in ("vectoriel", "graphe"):
    sous = [r for r in reponses if r["pipeline"] == pipeline]
    f = sum(r["fidelite"] for r in sous if r["fidelite"] is not None) / len(sous)
    p = sum(r["pertinence"] for r in sous if r["pertinence"] is not None) / len(sous)
    e = sum(r["correct"] for r in sous) / len(sous)
    print(f"{pipeline:<14}{f:>11.2f}{p:>13.2f}{e:>13.2f}")

print("\n=== STABILITÉ DU JUGE ===")
ef = [r["fidelite_ecart"] for r in reponses]
ep = [r["pertinence_ecart"] for r in reponses]
print(f"Écart max entre 3 jugements — fidélité   : "
      f"moyen {sum(ef)/len(ef):.2f}, maximum {max(ef):.2f}")
print(f"Écart max entre 3 jugements — pertinence : "
      f"moyen {sum(ep)/len(ep):.2f}, maximum {max(ep):.2f}")
print(f"Jugements parfaitement stables : "
      f"{sum(1 for x in ef if x == 0)}/{len(ef)} (fidélité), "
      f"{sum(1 for x in ep if x == 0)}/{len(ep)} (pertinence)")

print("\n=== PAR CATÉGORIE ===")
for genre in ("single-hop", "pont", "intersection"):
    print(f"\n{genre}")
    for pipeline in ("vectoriel", "graphe"):
        sous = [r for r in reponses
                if r["pipeline"] == pipeline and r["type"] == genre]
        if sous:
            f = sum(r["fidelite"] for r in sous if r["fidelite"] is not None) / len(sous)
            p = sum(r["pertinence"] for r in sous if r["pertinence"] is not None) / len(sous)
            print(f"  {pipeline:<12} fidélité {f:.2f}   pertinence {p:.2f}")

Modèle chargé.

  10/40
  20/40
  30/40
  40/40

                 fidélité   pertinence   exactitude
vectoriel            0.86         0.60         0.30
graphe               0.81         0.76         0.70

=== STABILITÉ DU JUGE ===
Écart max entre 3 jugements — fidélité   : moyen 0.01, maximum 0.25
Écart max entre 3 jugements — pertinence : moyen 0.00, maximum 0.00
Jugements parfaitement stables : 39/40 (fidélité), 40/40 (pertinence)

=== PAR CATÉGORIE ===

single-hop
  vectoriel    fidélité 0.81   pertinence 0.69
  graphe       fidélité 0.81   pertinence 0.86

pont
  vectoriel    fidélité 0.92   pertinence 0.42
  graphe       fidélité 0.75   pertinence 0.58

intersection
  vectoriel    fidélité 0.91   pertinence 0.56
  graphe       fidélité 0.83   pertinence 0.72
